<a href="https://colab.research.google.com/github/jollyoli93/jackoliver_24862664_dissertation/blob/main/YoloV8_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Start Up

In [ ]:
!git clone https://github.com/jollyoli93/jackoliver_24862664_dissertation

In [ ]:
!ls jackoliver_24862664_dissertation/

In [ ]:
yolov8n_path = '/content/jackoliver_24862664_dissertation/yolov8n'
yolov8s_path = '/content/jackoliver_24862664_dissertation/yolov8s'

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Ultralytics YOLOv8 Citation

@software{yolov8_ultralytics,
  author = {Glenn Jocher and Ayush Chaurasia and Jing Qiu},
  title = {Ultralytics YOLOv8},
  version = {8.0.0},
  year = {2023},
  url = {https://github.com/ultralytics/ultralytics},
  orcid = {0000-0001-5950-6979, 0000-0002-7603-6750, 0000-0003-3783-7069},
  license = {AGPL-3.0}
}

In [ ]:
import pandas as pd
import random
from sklearn.model_selection import KFold



In [ ]:
!pip install --upgrade ultralytics==8.3 wandb

In [ ]:
# import wandb

In [ ]:
# wandb.login()

In [ ]:
# from wandb.integration.ultralytics import add_wandb_callback

from ultralytics import YOLO

## For Extracting YOLO ZIP Files


Download '80_20_split.zip' and upload directly into /content/ folder (>1GB)

https://stummuac-my.sharepoint.com/:f:/g/personal/24862664_stu_mmu_ac_uk/IgDWqhOlnmoYSpLQ6sAMCdB5ARyt1K-IxLJHMlazY1jhaLU?e=ZZlzCj

In [ ]:
!unzip 80_20_split.zip -d /content/

In [ ]:
file_dir = '/content/80_20_split'

In [ ]:
!ls

# Define training step

In [ ]:
import torch
import gc

def train_one_model(MODEL, img_size, epochs, dataset_yaml, project, name, batch=16, workers=2, rect=False, val=True):
  model = YOLO(MODEL, task="detect")
  model.train(
      data=dataset_yaml, epochs=epochs, batch=batch, imgsz=img_size, rect=rect, project=project, workers=workers, name=name, val=val
  )  # include any additional train arguments,
  model.val()
  gc.collect()

  # return res

# Run Test/Valid

## 80/20 split no YOLOv8n

In [ ]:
IMG_SIZES = [512, 1024, 2048, 2560]

batch = 28
epochs = 100
project = "test"
workers = 4
ds = '/content/80_20_split/data.yaml'


In [ ]:
for img_size in IMG_SIZES:
  project = "yolov8n_imgsz"
  name = f"yolov8n_{img_size}.1"
  train_one_model(MODEL=yolov8n_path, dataset_yaml=ds, img_size=img_size, epochs=epochs, project=project, name=name, batch=batch, workers=workers)

In [ ]:
name = "test_512.2"
result =
train_one_model(MODEL=yolov8n_path, dataset_yaml=ds, img_size=512, epochs=1, project=project, name=name, batch=batch, workers=workers)

In [ ]:
result

## 80/20 split for yolov8s

In [ ]:
yolov8s = YOLO("yolov8s.pt")
IMG_SIZES = [512, 1024, 2048, 2560]

batch = 28
epochs = 100
project = "test_v8s"
workers = 4
ds = '/content/80_20_split/data.yaml'


In [ ]:
for img_size in IMG_SIZES:
  project = "yolov8s_imgsz"
  name = f"yolov8s_{img_size}"
  train_one_model(MODEL=yolov8s_path, dataset_yaml=ds, img_size=img_size, epochs=epochs, project=project, name=name, batch=batch, workers=workers)

# K_Fold Split

https://docs.ultralytics.com/guides/kfold-cross-validation

In [ ]:
zip_path = "/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/orgsize-k_fold.yolov8.zip"
extract_path = "/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")

In [ ]:
# !pip install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="MYzeeHsKmZnJ360DkKYF")
# project = rf.workspace("masterproject-mvazq").project("master_project-tn4xh")
# version = project.version(26)
# dataset = version.download("yolov8")


In [ ]:
from pathlib import Path

dataset_path = Path("/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/train")
labels = sorted(dataset_path.rglob("*labels/*.txt"))  # all data in 'labels'
images = list(dataset_path.rglob("*images/*.jpg"))

In [ ]:
import yaml

yaml_file = "/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/data.yaml"  # your data YAML with data directories and names dictionary
with open(yaml_file, encoding="utf8") as y:
    classes = yaml.safe_load(y)["names"]
cls_idx = list(range(len(classes)))

In [ ]:
import pandas as pd

index = [label.stem for label in labels]  # uses base filename as ID (no extension)
labels_df = pd.DataFrame([], columns=cls_idx, index=index)

In [ ]:
from collections import Counter

for label in labels:
    lbl_counter = Counter()

    with open(label) as lf:
        lines = lf.readlines()

    for line in lines:
        # classes for YOLO label uses integer at first position of each line
        lbl_counter[int(line.split(" ", 1)[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0)  # replace `nan` values with `0.0`

In [ ]:
%cd /content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold

In [ ]:
pd.DataFrame.to_csv(labels_df, "./dataframe.csv")

In [ ]:
random.seed(0)  # for reproducibility
ksplit = 5
kf = KFold(n_splits=ksplit, shuffle=True, random_state=20)  # setting random_state for repeatable results

kfolds = list(kf.split(labels_df)) #list of lists

In [ ]:
k_test, k_train = kfolds[0]

In [ ]:
train_tester = [0,1,2]
labels_df.iloc[train_tester].index #converts index to labels as K_Fold is list of index

In [ ]:
folds = [f"split_{n}" for n in range(1, ksplit + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(kfolds, start=1):
    folds_df[f"split_{i}"].loc[labels_df.iloc[train].index] = "train"
    folds_df[f"split_{i}"].loc[labels_df.iloc[val].index] = "val"

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio

In [ ]:
import datetime

supported_extensions = [".jpg", ".jpeg", ".png"]

# Initialize an empty list to store image file paths
images = []

# Loop through supported extensions and gather image files
for ext in supported_extensions:
    images.extend(sorted((dataset_path / "images").rglob(f"*{ext}")))

# Create the necessary directories and dataset YAML files
save_path = Path(dataset_path / f"{datetime.date.today().isoformat()}_{ksplit}-Fold_Cross-val")
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": classes,
            },
            ds_y,
        )

### Copy all images into relevent folders

In [ ]:
import shutil

from tqdm import tqdm

for image, label in tqdm(zip(images, labels), total=len(images), desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"

        # Copy image and label files to new directory (SamefileError if file already exists)
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

In [ ]:
folds_df.to_csv(save_path / "kfold_datasplit.csv")
fold_lbl_distrb.to_csv(save_path / "kfold_label_distribution.csv")

## Load K_Fold split data

In [ ]:
base_dir = "/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/train/2026-01-26_5-Fold_Cross-val"

In [ ]:
import pandas as pd

folds_df = pd.read_csv(base_dir + "/kfold_datasplit.csv")
fold_lbl_distrb = pd.read_csv(base_dir + "/kfold_label_distribution.csv")

In [ ]:
ds_yamls = [f"{base_dir}/split_{i}/split_{i}_dataset.yaml" for i in range(1,6) ]

In [ ]:
ds_yamls

# Model Config

In [ ]:
import gc

def K_Fold(model):
  for k, ds in enumerate(ds_yamls):
    name =f"fold_{k + 1}"

    res = train_one_model(MODEL=model, dataset_yaml=ds, img_size=IMG_SIZE, epochs=epochs, project=project, name=name, batch=batch, workers=workers)

    results[k] = {
        "save_dir": getattr(res, "save_dir", None)
    }

    del res
    del model
    gc.collect()
    torch.cuda.empty_cache()

## Run YOLOv8n K-Fold

In [ ]:
results = {}

# yolov8n = "/content/drive/MyDrive/Crane-Detector/weights/yolov8n.pt" #use directory to prevent default YOLOv11
IMG_SIZE = 2048

batch = 28
epochs = 100
project = "test" #k_fold_yolov8n"
workers = 4

In [ ]:
K_Fold(yolov8n_path)

## Run YOLOv8s K-Fold

In [ ]:
# %cd '/content/drive/MyDrive/Crane-Detector/YOLOv8 Normal/K_Fold/k_fold_yolov8s'

In [ ]:
!ls

In [ ]:
results_s = {}

yolov8s = "yolov8s.pt"
IMG_SIZE = 2048

batch = 28
epochs = 100
project = "k_fold_yolov8s"
workers = 4

In [ ]:
K_Fold(yolov8s_path)